# Whisper 학습 노트북

이 노트북은 OpenAI Whisper 모델을 한국어 GOLD JSONL 로 파인튜닝하기 위한 셀 골격입니다.

## 사용 흐름

1. 환경 점검 (셀 1~3)
2. 데이터 로드 + 검증 (셀 4~5)
3. Whisper Dataset 어댑터 (셀 6)
4. 학습 (셀 7~8) — HuggingFace Trainer 노트북 내 직접 실행
5. 평가 (셀 9)

## 약속

- 핵심 로직은 `project/` 모듈에. 노트북은 *호출만*.
- 모든 경로/백본/W&B 는 `configs/<exp>.yaml` 에서 읽는다.

## 셀 1 — 환경 점검

In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO = Path('/home/cssong/workspace/TRAIN-ASR')
sys.path.insert(0, str(REPO))
os.chdir(REPO)

print('CONDA_DEFAULT_ENV:', os.environ.get('CONDA_DEFAULT_ENV'))
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total', '--format=csv'], check=False)

## 셀 2 — Config 로드

In [ ]:
from project.utils.config import load_config
from project.utils.seed import seed_everything

CFG_PATH = REPO / 'configs/default.yaml'
cfg = load_config(CFG_PATH)
seed_everything(cfg['experiment']['seed'])
print('experiment:', cfg['experiment']['name'])
print('backbone:  ', cfg['models']['whisper']['backbone'])
print('train data:', cfg['paths']['train_jsonl'])

## 셀 3 — 의존성 sanity

In [ ]:
import importlib

needed = ['transformers', 'datasets', 'accelerate', 'evaluate', 'jiwer', 'soundfile', 'torch']
for pkg in needed:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'  {pkg:14s} {v}')
    except ImportError:
        print(f'  {pkg:14s} ❌ 미설치')

import torch
print('CUDA available:', torch.cuda.is_available(), '| device count:', torch.cuda.device_count())

GOLD JSONL 로드 + 자동 검증. 실패 시 `SchemaValidationError`.

In [ ]:
from project.data import load_samples

train = load_samples(cfg['paths']['train_jsonl'])
val   = load_samples(cfg['paths']['val_jsonl'])
print(f'train: {len(train):,}  /  val: {len(val):,}')
print('sample[0]:', train[0])

## 셀 5 — 빠른 분포 점검 (SV 노트북과 동일)

In [ ]:
from collections import Counter

print(f'화자 수: {len({s.speaker_id for s in train}):,}')
print(f'코퍼스: {Counter(s.corpus_id for s in train).most_common()}')
print(f'gender: {Counter(s.gender for s in train).most_common()}')
print(f'age:    {Counter(s.age_group for s in train).most_common()}')

## 셀 6 — Whisper Dataset 어댑터 (GOLD → HuggingFace Dataset)

어댑터 로직은 `project/data/adapters/whisper.py` 에 둘 것 (구현 예정).

In [ ]:
# TODO: 어댑터 구현 후 활성화
# from project.data.adapters.whisper import to_whisper_dataset
#
# train_ds = to_whisper_dataset(train, backbone=cfg['models']['whisper']['backbone'])
# val_ds   = to_whisper_dataset(val,   backbone=cfg['models']['whisper']['backbone'])
# print(train_ds)

print('어댑터 구현 후 활성화 — project/data/adapters/whisper.py')

## 셀 7 — 모델 / Trainer 빌드 (HF Trainer)

In [ ]:
# TODO: 학습 모듈 구현 후 활성화
# from project.training.whisper import build_trainer
#
# trainer = build_trainer(
#     cfg=cfg,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
# )
# print(trainer.args.output_dir)

print('학습 모듈 구현 후 활성화 — project/training/whisper.py')

## 셀 8 — 학습 실행

노트북 셀 내에서 실행. 단, *장시간 학습은 tmux/screen* 외부 실행 권장.

In [ ]:
# trainer.train()
# trainer.save_model()

print('build_trainer() 후 활성화')

## 셀 9 — 학습 후 평가 (SV 노트북과 동일 인터페이스)

결과 포맷은 두 모델 공통: `evaluation_report.{txt,json}` + `<run>_diff.txt`.
참고 양식: `practice/2601_sensevoice_train/results/finetuned_*/`.

In [ ]:
# TODO: 평가 모듈 구현 후 활성화
# from project.evaluation import evaluate_on_benchmark_suite
#
# report = evaluate_on_benchmark_suite(
#     model_path=trainer.args.output_dir,
#     model_type='whisper',
#     benchmarks=cfg['eval']['benchmarks'],
#     out_dir=Path(cfg['paths']['eval_results_dir']) / cfg['experiment']['name'],
# )
# print(report)

print('평가 모듈 구현 후 활성화 — project/evaluation/')